# Notebook 1: Data Collection & Cleaning
## Clustering Premier League Playstyles & Predicting Match Outcomes
**ISYE 6740 | Summer 2026 | Group 078**

This notebook uses the [`fbrefdata`](https://github.com/lorenzodb1/fbrefdata) package to collect match-level team stats from FBref. It's a lightweight, requests-based scraper (no Selenium/Chrome needed) that handles caching, rate limiting, and returns clean DataFrames.

### Data Collected
| Category | Stat Type | Key Features |
|----------|-----------|-------------|
| Schedule | `schedule` | Date, venue, result, possession %, GF, GA |
| Shooting | `shooting` | Shots, SoT, xG, distance |
| Passing | `passing` | Completion %, progressive passes, key passes |
| Possession | `possession` | Touches, carries, progressive carries |
| Defense | `defense` | Tackles, interceptions, blocks, clearances |
| Misc | `misc` | Fouls, cards, aerials won/lost, recoveries |
| GCA | `goal_shot_creation` | Shot-creating & goal-creating actions |

### Leagues & Seasons
- **EPL**: 2023/24, 2024/25
- **La Liga**: 2023/24, 2024/25
- **~3,040 team-match observations** total

## 0. Setup

Run this cell once to install dependencies.

In [ ]:
# Install fbrefdata (requests-based, no Chrome/Selenium needed)
!pip install fbrefdata pandas numpy tqdm

In [ ]:
import fbrefdata as fd
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f"fbrefdata version: {fd.__version__}")
print("All imports successful.")

## 1. Configure fbrefdata

`fbrefdata` needs a `league_dict.json` config to know which leagues to recognize. This cell creates it automatically.

In [ ]:
# Create fbrefdata config directory and league mapping
config_dir = Path.home() / "fbrefdata" / "config"
config_dir.mkdir(parents=True, exist_ok=True)

league_dict = {
    "ENG-Premier League": {
        "FBref": "Premier League"
    },
    "SPA-La Liga": {
        "FBref": "La Liga"
    }
}

config_path = config_dir / "league_dict.json"
with open(config_path, 'w') as f:
    json.dump(league_dict, f, indent=2)

print(f"Config written to: {config_path}")
print(f"Configured leagues: {list(league_dict.keys())}")

In [ ]:
# Verify fbrefdata recognizes our leagues
print("Available leagues:", fd.FBref.available_leagues())

## 2. Configuration

In [ ]:
# === PROJECT CONFIGURATION ===

LEAGUES = ["ENG-Premier League", "SPA-La Liga"]
SEASONS = [2024, 2025]  # fbrefdata convention: 2024 = 2023-24 season, 2025 = 2024-25

# All available match-log stat types in fbrefdata
STAT_TYPES = [
    "schedule",              # Date, venue, result, possession, GF, GA
    "shooting",              # Shots, SoT, xG, FK, PK
    "passing",               # Completion %, progressive passes, key passes
    "possession",            # Touches, carries, progressive carries
    "defense",               # Tackles, interceptions, blocks
    "misc",                  # Fouls, cards, aerials, recoveries
    "goal_shot_creation",    # SCA, GCA
]

DATA_DIR = os.path.join("..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Leagues: {LEAGUES}")
print(f"Seasons: {SEASONS}")
print(f"Stat types: {len(STAT_TYPES)}")
print(f"Expected observations: ~{len(SEASONS) * len(LEAGUES) * 20 * 38} team-match rows")

## 3. Scrape Match-Level Data

`fbrefdata` handles rate limiting and caches data locally in `~/fbrefdata/data/`. First run takes **~30-45 minutes** (FBref rate limits to ~20 req/min). Subsequent runs are instant from cache.

We scrape each stat type separately, then merge on date + team.

In [ ]:
def scrape_league_season(league, season, stat_types):
    """
    Scrape all stat types for a league-season and merge into a single DataFrame.
    
    Each stat type returns a DataFrame indexed by (league, season, team, game).
    We merge them on this multi-index to get one wide row per team-match.
    """
    print(f"\n{'='*60}")
    print(f"Scraping: {league} {season}")
    print(f"{'='*60}")
    
    fbref = fd.FBref(leagues=league, seasons=season)
    
    dfs = {}
    for stat_type in stat_types:
        print(f"  Fetching {stat_type}...")
        try:
            df = fbref.read_team_match_stats(stat_type=stat_type)
            
            # Flatten multi-level columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [
                    f"{stat_type}_{lvl1}_{lvl2}".strip('_') 
                    if lvl1 != '' and lvl2 != '' 
                    else f"{stat_type}_{lvl1 or lvl2}"
                    for lvl1, lvl2 in df.columns
                ]
            else:
                # Prefix columns with stat_type to avoid collisions
                df.columns = [f"{stat_type}_{c}" for c in df.columns]
            
            dfs[stat_type] = df
            print(f"    → {df.shape[0]} rows, {df.shape[1]} columns")
        except Exception as e:
            print(f"    ✗ Failed: {e}")
    
    if not dfs:
        print("  ERROR: No data scraped!")
        return pd.DataFrame()
    
    # Merge all stat types on the shared index (league, season, team, game)
    merged = None
    for stat_type, df in dfs.items():
        if merged is None:
            merged = df
        else:
            # Join on index — all DataFrames share the same multi-index
            merged = merged.join(df, how='outer')
    
    print(f"\n  Merged: {merged.shape[0]} rows × {merged.shape[1]} columns")
    return merged


print("Scraping function defined.")

In [ ]:
# ============================================================
# SCRAPE ALL DATA
# ============================================================
# First run: ~30-45 minutes (FBref rate limiting)
# Subsequent runs: instant (cached in ~/fbrefdata/data/)
#
# If you get rate-limited (429 errors), wait a few minutes and re-run.
# fbrefdata will resume from cache for already-downloaded pages.
# ============================================================

all_data = []

for league in LEAGUES:
    for season in SEASONS:
        df = scrape_league_season(league, season, STAT_TYPES)
        if not df.empty:
            all_data.append(df)

# Combine everything
raw_df = pd.concat(all_data)
print(f"\n{'='*60}")
print(f"TOTAL: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns")
print(f"{'='*60}")

# Save raw data
raw_path = os.path.join(DATA_DIR, "raw_match_logs.csv")
raw_df.to_csv(raw_path)
print(f"\nSaved raw data to {raw_path}")

In [ ]:
# Quick look at what we got
print(f"Index levels: {raw_df.index.names}")
print(f"\nAll columns ({len(raw_df.columns)}):")
for col in sorted(raw_df.columns):
    print(f"  {col}")

## 4. Data Cleaning & Feature Engineering

Map the scraped columns to our target feature set and compute derived features.

In [ ]:
# Reset index to make league/season/team/game regular columns
df = raw_df.reset_index()

# Standardize column names
df.columns = (
    df.columns
    .str.lower()
    .str.replace(r'[^a-z0-9_]', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)

print(f"Cleaned columns: {len(df.columns)}")
print(f"\nSample columns:")
for col in df.columns[:30]:
    print(f"  {col}")

In [ ]:
def find_column(df, *keywords):
    """
    Find a column whose name contains all given keywords.
    Returns the first match or None.
    """
    for col in df.columns:
        if all(kw in col for kw in keywords):
            return col
    return None


def build_feature_mapping(df):
    """
    Auto-discover column mappings from scraped data.
    Prints what was found and what's missing so you can adjust.
    """
    # Target features → search patterns
    targets = {
        # --- Identifiers ---
        'date':                 [('schedule', 'date')],
        'squad':                [('team',)],
        'opponent':             [('schedule', 'opponent'), ('schedule', 'opp')],
        'venue':                [('schedule', 'venue')],
        'result':               [('schedule', 'result')],
        'goals_for':            [('schedule', 'gf'), ('schedule', 'goals_for')],
        'goals_against':        [('schedule', 'ga'), ('schedule', 'goals_against')],
        
        # --- Possession & Passing ---
        'possession_pct':       [('schedule', 'poss'), ('possession', 'poss')],
        'pass_completion_pct':  [('passing', 'cmp_pct'), ('passing', 'cmp%'), ('passing', 'total_cmp_pct'), ('passing', 'total_cmp_percent')],
        'progressive_passes':   [('passing', 'prgp'), ('passing', 'prog')],
        'progressive_carries':  [('possession', 'prgc'), ('possession', 'prog')],
        'carries':              [('possession', 'carries_carries'), ('possession', 'carries')],
        
        # --- Shooting ---
        'shots':                [('shooting', 'standard_sh'), ('shooting', 'sh')],
        'shots_on_target':      [('shooting', 'standard_sot'), ('shooting', 'sot')],
        'xg':                   [('shooting', 'expected_xg'), ('shooting', 'xg'), ('schedule', 'xg')],
        
        # --- Pressing / Defense ---
        'pressures':            [('defense', 'pressures_press'), ('defense', 'press')],
        'successful_pressures': [('defense', 'pressures_succ'), ('defense', 'succ')],
        'tackles':              [('defense', 'tackles_tkl'), ('defense', 'tkl')],
        'interceptions':        [('defense', 'int'), ('defense', 'interceptions')],
        'blocks':               [('defense', 'blocks_blocks'), ('defense', 'blocks')],
        'clearances':           [('defense', 'clr'), ('defense', 'clearances')],
        
        # --- Aerials ---
        'aerials_won':          [('misc', 'aerial_duels_won'), ('misc', 'won')],
        'aerials_lost':         [('misc', 'aerial_duels_lost'), ('misc', 'lost')],
        
        # --- Set Pieces ---
        'corners':              [('passing_types', 'ck'), ('passing', 'ck'), ('misc', 'ck')],
        'crosses_into_box':     [('passing', 'crspa'), ('passing_types', 'crs')],
        
        # --- GCA ---
        'shot_creating_actions': [('goal_shot_creation', 'sca_sca'), ('goal_shot_creation', 'sca')],
        'goal_creating_actions': [('goal_shot_creation', 'gca_gca'), ('goal_shot_creation', 'gca')],
    }
    
    mapping = {}  # scraped_col → clean_name
    found = []
    missing = []
    
    for target_name, search_patterns in targets.items():
        matched = False
        for pattern in search_patterns:
            col = find_column(df, *pattern)
            if col:
                mapping[col] = target_name
                found.append((target_name, col))
                matched = True
                break
        if not matched:
            missing.append(target_name)
    
    print(f"Mapped {len(found)}/{len(targets)} target features:")
    for target, source in found:
        print(f"  ✓ {target:30s} ← {source}")
    
    if missing:
        print(f"\nMissing {len(missing)} features (check column names manually):")
        for m in missing:
            print(f"  ✗ {m}")
    
    return mapping


col_mapping = build_feature_mapping(df)

In [ ]:
# ============================================================
# MANUAL FIXES
# ============================================================
# If any features were missing above, find the correct column names
# by searching through df.columns and add them here:
#
# col_mapping['actual_column_name_in_df'] = 'target_feature_name'
#
# Example:
# col_mapping['defense_tackles_tkl_def_3rd'] = 'tackles'  # if auto-match failed
# ============================================================

# Uncomment and adjust as needed after inspecting the output above:
# col_mapping['some_column'] = 'some_feature'

print(f"Total mapped columns: {len(col_mapping)}")

In [ ]:
def clean_and_engineer(df, col_mapping):
    """
    Apply column mapping, compute derived features, handle missing values.
    """
    # Keep all columns but rename the mapped ones
    clean = df.rename(columns=col_mapping)
    
    # --- Derived features ---
    
    # Pressing intensity = successful pressures / total pressures
    if 'pressures' in clean.columns and 'successful_pressures' in clean.columns:
        clean['pressures'] = pd.to_numeric(clean['pressures'], errors='coerce')
        clean['successful_pressures'] = pd.to_numeric(clean['successful_pressures'], errors='coerce')
        clean['pressing_intensity'] = (
            clean['successful_pressures'] / clean['pressures'].replace(0, np.nan)
        )
    
    # Aerial win rate
    if 'aerials_won' in clean.columns and 'aerials_lost' in clean.columns:
        clean['aerials_won'] = pd.to_numeric(clean['aerials_won'], errors='coerce')
        clean['aerials_lost'] = pd.to_numeric(clean['aerials_lost'], errors='coerce')
        total_aerials = clean['aerials_won'] + clean['aerials_lost']
        clean['aerial_win_rate'] = clean['aerials_won'] / total_aerials.replace(0, np.nan)
    
    # --- Convert numeric columns ---
    exclude_cols = {'date', 'squad', 'opponent', 'venue', 'result', 
                    'league', 'season', 'game', 'match_report', 'notes',
                    'day', 'time', 'round', 'referee', 'captain',
                    'formation', 'comp'}
    for col in clean.columns:
        if col.lower() not in exclude_cols:
            clean[col] = pd.to_numeric(clean[col], errors='coerce')
    
    # --- Parse date ---
    if 'date' in clean.columns:
        clean['date'] = pd.to_datetime(clean['date'], errors='coerce')
        clean = clean.sort_values(['league', 'season', 'squad', 'date'])
    
    # --- Rolling form (points over last 5 matches) ---
    if 'result' in clean.columns:
        points_map = {'W': 3, 'D': 1, 'L': 0}
        clean['match_points'] = clean['result'].map(points_map)
        clean['rolling_form_5'] = (
            clean.groupby(['league', 'season', 'squad'])['match_points']
            .transform(lambda x: x.rolling(5, min_periods=1).mean())
        )
    
    # --- Handle missing values ---
    missing_pct = clean.isnull().mean().sort_values(ascending=False)
    significant = missing_pct[missing_pct > 0.05]
    if not significant.empty:
        print(f"Columns with >5% missing:")
        for col, pct in significant.head(15).items():
            print(f"  {col}: {pct:.1%}")
    
    # Drop columns with >50% missing
    drop_cols = missing_pct[missing_pct > 0.50].index
    if len(drop_cols) > 0:
        print(f"\nDropping {len(drop_cols)} columns with >50% missing")
        clean = clean.drop(columns=drop_cols)
    
    # Fill remaining NaN with column median for numeric columns
    num_cols = clean.select_dtypes(include=[np.number]).columns
    clean[num_cols] = clean[num_cols].fillna(clean[num_cols].median())
    
    return clean


clean_df = clean_and_engineer(df, col_mapping)
print(f"\nCleaned dataset: {clean_df.shape}")
clean_df.head(3)

## 5. Select Clustering Features & Export

In [ ]:
# Features for playstyle clustering (from proposal Section 3)
CLUSTERING_FEATURES = [
    'possession_pct',
    'pass_completion_pct',
    'progressive_passes',
    'progressive_carries',
    'shots',
    'shots_on_target',
    'xg',
    'pressures',
    'successful_pressures',
    'pressing_intensity',
    'tackles',
    'interceptions',
    'aerial_win_rate',
    'crosses_into_box',
    'corners',
    'carries',
    'blocks',
    'clearances',
]

# Check which features we actually have
available = [f for f in CLUSTERING_FEATURES if f in clean_df.columns]
missing = [f for f in CLUSTERING_FEATURES if f not in clean_df.columns]

print(f"Available clustering features: {len(available)}/{len(CLUSTERING_FEATURES)}")
for f in available:
    print(f"  ✓ {f}")

if missing:
    print(f"\nMissing features:")
    for f in missing:
        print(f"  ✗ {f}")
    print("\n  → Check col_mapping in cell above and add manual fixes")

In [ ]:
# Metadata columns to keep
META_COLS = ['date', 'squad', 'opponent', 'venue', 'result',
             'goals_for', 'goals_against', 'league', 'season',
             'rolling_form_5', 'match_points']

available_meta = [c for c in META_COLS if c in clean_df.columns]

# Build final dataset
final_df = clean_df[available_meta + available].copy()
final_df = final_df.dropna(subset=['date', 'squad'])  # drop any junk rows

print(f"Final dataset: {final_df.shape}")
print(f"\nLeagues: {final_df['league'].unique()}")
print(f"Seasons: {final_df['season'].unique()}")
print(f"\nTeams per league-season:")
print(final_df.groupby(['league', 'season'])['squad'].nunique())
print(f"\nMatches per team (mean):")
print(final_df.groupby(['league', 'season'])['squad'].value_counts().groupby(level=[0,1]).mean().round(1))

In [ ]:
# Feature summary statistics
print("Feature summary:")
final_df[available].describe().round(2)

In [ ]:
# ============================================================
# EXPORT
# ============================================================

# Full dataset
final_df.to_csv(os.path.join(DATA_DIR, "match_data_clean.csv"), index=False)

# League splits
final_df[final_df['league'] == 'ENG-Premier League'].to_csv(
    os.path.join(DATA_DIR, "epl_match_data.csv"), index=False)
final_df[final_df['league'] == 'SPA-La Liga'].to_csv(
    os.path.join(DATA_DIR, "laliga_match_data.csv"), index=False)

# Feature list for downstream notebooks
with open(os.path.join(DATA_DIR, "feature_columns.json"), 'w') as f:
    json.dump(available, f, indent=2)

print("Exported:")
print(f"  data/match_data_clean.csv     ({len(final_df)} rows)")
print(f"  data/epl_match_data.csv       ({(final_df['league'] == 'ENG-Premier League').sum()} rows)")
print(f"  data/laliga_match_data.csv    ({(final_df['league'] == 'SPA-La Liga').sum()} rows)")
print(f"  data/feature_columns.json     ({len(available)} features)")

## 6. Data Validation

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)

for league in final_df['league'].unique():
    for season in final_df['season'].unique():
        subset = final_df[(final_df['league'] == league) & (final_df['season'] == season)]
        n_teams = subset['squad'].nunique()
        matches_per_team = subset.groupby('squad').size()
        
        print(f"\n{league} {season}:")
        print(f"  Teams: {n_teams} (expected: 20)")
        print(f"  Matches/team: min={matches_per_team.min()}, "
              f"max={matches_per_team.max()}, "
              f"mean={matches_per_team.mean():.1f} (expected: 38)")
        
        dupes = subset.duplicated(subset=['squad', 'date']).sum()
        print(f"  Duplicates: {dupes} {'✓' if dupes == 0 else '⚠'}")

total_missing = final_df[available].isnull().sum().sum()
total_cells = len(final_df) * len(available)
print(f"\nOverall missing: {total_missing}/{total_cells} ({total_missing/total_cells:.2%})")
print("=" * 50)

---
## Next Steps

1. Verify the CSVs in `data/`
2. Proceed to **`02_eda_clustering.ipynb`** for PCA and GMM clustering
3. Then **`03_prediction_crossleague.ipynb`** for match outcome prediction

### Troubleshooting
- **Rate limited (429)?** Wait 5 minutes and re-run — cached pages won't re-download
- **Missing features?** Print `df.columns.tolist()` and update `col_mapping` manually
- **Wrong league names?** Check `~/fbrefdata/config/league_dict.json`
- **Want to force re-scrape?** Delete `~/fbrefdata/data/` and re-run